### Simulation: Single ETF Timing Strategy

**Portfolio:** 100% SPY, 0% QQQ, 0% VTI <br>
**Strategy:** Buy shares on day 1, sells shares when moving average falls below the current ETF price, then buy more shares with stored cash when moving average is greater than the current price <br>
**Initial Investment:** $10,000 <br>
**Biweekly Contribution:** Add $500 every 10 trading days and buy more shares. <br>

#### 1. Import Libaries and Load Dataset

In [22]:
import pandas as pd
import numpy as np
import os 

PROCESSED_DIR   = "../data/processed"
SIMULATIONS_DIR = "../data/simulations"
os.makedirs(SIMULATIONS_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PROCESSED_DIR, "prices_with_indicators.csv"))
df.head()

,Date,SPY_Price,QQQ_Price,VTI_Price,SPY_MA50,QQQ_MA50,VTI_MA50
0,2001-06-15,77.995598,35.981022,36.027668,NaN,NaN,NaN
1,2001-06-18,77.617950,35.499580,35.797894,NaN,NaN,NaN
2,2001-06-19,77.957207,35.322208,35.898216,NaN,NaN,NaN
3,2001-06-20,78.366867,36.124603,36.276844,NaN,NaN,NaN
4,2001-06-21,79.256599,36.614502,36.568089,NaN,NaN,NaN


simple cleaning to ensure columns are correct

In [23]:
# columns that should be numeric
numeric_cols = [
    "SPY_Price", "QQQ_Price", "VTI_Price",
    "SPY_MA50", "QQQ_MA50", "VTI_MA50"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("`", "", regex=False)   # remove backticks
        .str.replace("'", "", regex=False)   # remove apostrophes too, just in case
        .str.replace(",", "", regex=False)   # remove commas if any
        .str.strip()
    )

    # convert cleaned text back to numeric
    df[col] = pd.to_numeric(df[col], errors="coerce")

### 2. Initialize Variables

In [24]:
initial_investment = 10000
biweekly_contribution = 500   # or 200 if that is your project value

first_price = df.loc[0, "SPY_Price"]

shares_spy = initial_investment / first_price
cash = 0
total_invested = initial_investment

rows = []
previous_portfolio_value = None

### 3. Start Simulation

In [25]:
for i in range(len(df)):
    
    source_row = df.loc[i]
    
    #pull values from source dataset
    date = source_row["Date"]
    spy_price = source_row["SPY_Price"]
    qqq_price = source_row["QQQ_Price"]
    vti_price = source_row["VTI_Price"]
    
    spy_ma50 = source_row["SPY_MA50"]
    qqq_ma50 = source_row["QQQ_MA50"]
    vti_ma50 = source_row["VTI_MA50"]
    
    #default values for this day
    contribution = 0
    position_spy = 0
    position_qqq = 0
    position_vti = 0
    
    #biweekly contribution
    if i % 10 == 0 and i != 0:
        contribution = biweekly_contribution
        cash += contribution
        total_invested += contribution
        
    #check to buy or sell shares
    if pd.notna(spy_ma50):
        
        #buy shares and stay invested
        if spy_price > spy_ma50:
            position_spy = 1
            
            if cash > 0:
                shares_spy += cash / spy_price
                cash = 0
        
        #sell and put into cash
        else:
            position_spy = 0
            
            if shares_spy > 0:
                cash += shares_spy * spy_price
                shares_spy = 0
    
    else:
        position_spy = 1 if shares_spy > 0 else 0

        if cash > 0:
            shares_spy += cash / spy_price
            cash = 0
            
    #calculate portfolio value
    portfolio_value = (shares_spy * spy_price) + cash
    earnings = portfolio_value - total_invested
    
    #calculate daily return
    if previous_portfolio_value is None:
        daily_return = 0
    else:
        daily_return = (portfolio_value - previous_portfolio_value) / previous_portfolio_value

    previous_portfolio_value = portfolio_value
    
    #store results for this day
    row = {
        "Date": date,
        "SPY_Price": spy_price,
        "QQQ_Price": qqq_price,
        "VTI_Price": vti_price,
        
        "SPY_MA50": spy_ma50,
        "QQQ_MA50": qqq_ma50,
        "VTI_MA50": vti_ma50,
        
        "Position_SPY": position_spy,
        "Position_QQQ": position_qqq,
        "Position_VTI": position_vti,
        
        "Shares_SPY": shares_spy,
        "Shares_QQQ": 0,
        "Shares_VTI": 0,
        
        "Cash": cash,
        "Contribution": contribution,
        "Total_Invested": total_invested,
        "Portfolio_Value": portfolio_value,
        "Earnings": earnings,
        "Daily_Return": daily_return,
        
        "Strategy": "Timing",
        "Portfolio_Type": "Single"
    }
    
    rows.append(row)

### 4. Data Clean and Validate

In [26]:
simulation_numeric_cols = [
    "SPY_Price", "QQQ_Price", "VTI_Price",
    "SPY_MA50", "QQQ_MA50", "VTI_MA50",
    "Position_SPY", "Position_QQQ", "Position_VTI",
    "Shares_SPY", "Shares_QQQ", "Shares_VTI",
    "Cash", "Contribution", "Total_Invested",
    "Portfolio_Value", "Earnings", "Daily_Return"
]

for col in simulation_numeric_cols:
    single_timing_df[col] = (
        single_timing_df[col]
        .astype(str)
        .str.replace("`", "", regex=False)
        .str.replace("'", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    single_timing_df[col] = pd.to_numeric(single_timing_df[col], errors="coerce")

In [27]:
print(df[numeric_cols].dtypes)
print(df[numeric_cols].isna().sum())

SPY_Price    float64
QQQ_Price    float64
VTI_Price    float64
SPY_MA50     float64
QQQ_MA50     float64
VTI_MA50     float64
dtype: object
SPY_Price     0
QQQ_Price     0
VTI_Price     0
SPY_MA50     49
QQQ_MA50     49
VTI_MA50     49
dtype: int64


### 5. Export Data

In [28]:
single_timing_df = pd.DataFrame(rows)
single_timing_df.head()

single_timing_df.to_csv(os.path.join(SIMULATIONS_DIR, "single_timing.csv"), index=False)